In [ ]:
import json
import re
from google import genai

In [ ]:
## 입력 PATHS
PDF_PATH = r"C:\Users\chiho\OneDrive\AI_PJTs\AIOCR\receipt\data\1.2025-11.pdf"  # 직원복무규정 PDF 파일 경로

## 출력 PATHS
JSON_PATH = r"C:\Users\chiho\OneDrive\AI_PJTs\AIOCR\receipt\output\llm_receipt_results.json"  # 직원복무규정 추출 JSON 파일 경로

## LLM MODEL 설정
MODEL_NAME =  "gemini-3-pro-preview"          # "gemini-2.5-pro"

In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [ ]:
# 1. 클라이언트 초기화
# 환경 변수에서 GEMINI_API_KEY를 자동으로 가져옵니다.
client = genai.Client()

# --- 2. File API를 사용하여 PDF 파일 업로드 ---
print(f"'{PDF_PATH}' 파일을 업로드하는 중...")

# 파일을 업로드하고 File 객체를 반환받습니다.
uploaded_file = client.files.upload(
    file=PDF_PATH
    # display_name=os.path.basename(PDF_FILE_PATH)
)
print(f"업로드 완료. 파일 이름: {uploaded_file.name}")
print(f"MIME 타입: {uploaded_file.mime_type}")

'C:\Users\chiho\OneDrive\AI_PJTs\AIOCR\receipt\data\1.2025-11.pdf' 파일을 업로드하는 중...
업로드 완료. 파일 이름: files/uxkgxw3n691r
MIME 타입: application/pdf


In [ ]:
# --- 3. Gemini 모델에 요청 보내기 ---
prompt = """
너는 전문 데이터 입력 사무원이야. 이 영수증 이미지를 보고 아래 요구사항에 맞춰 정보를 추출해줘.
    
    [요구사항]
    1. 가맹점명(상호), 거래일시(YYYY-MM-DD HH:MM:SS), 합계금액(숫자만), 품목리스트를 추출할 것.
    2. 날짜가 잘 안 보이면 '날짜 불명'으로 처리할 것.
    3. 결과는 반드시 오직 순수한 JSON 형식으로만 출력해. (마크다운 ```json 태그 쓰지 말 것)
    
    [JSON 출력 예시]
    {
        "store_name": "스타벅스 강남점",
        "date": "2025-11-05 12:30:00",
        "total_amount": 15000,
        "items": [
            {"name": "아메리카노", "price": 4500, "qty": 1},
            {"name": "카페라떼", "price": 5000, "qty": 2}
        ]
    }
    """

In [ ]:
# 모델에 업로드된 파일과 텍스트 프롬프트를 함께 전달합니다.
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[
        uploaded_file,
        prompt
    ]
)

# --- 4. 응답을 json 파일로 저장 ---
# Gemini API의 응답 텍스트를 가져옴
raw = response.text

# 정규식을 사용하여 마크다운 코드 블록(```json ... ``` 또는 ``` ... ```)을 제거
# re.DOTALL: 개행 문자를 포함한 모든 문자를 매칭
# re.IGNORECASE: 대소문자 구분 없이 매칭
m = re.match(r"^```(?:json)?\s*(.*)\s*```$", raw, flags=re.DOTALL | re.IGNORECASE)

# 매칭된 그룹(코드 블록 내부 내용)이 있으면 추출, 없으면 원본 텍스트 사용
clean = m.group(1) if m else raw


# Gemini 응답이 JSON 문자열이라면 바로 저장
parsed_json = json.loads(clean)
with open(JSON_PATH, "w", encoding="utf-8") as json_file:
    json.dump(parsed_json, json_file, ensure_ascii=False, indent=2)
print(f"\n결과가 '{JSON_PATH}' 파일로 저장되었습니다.")


결과가 'C:\Users\chiho\OneDrive\AI_PJTs\AIOCR\receipt\output\llm_receipt_results.json' 파일로 저장되었습니다.


In [ ]:
print(len(parsed_json))

16
